# Notebook 07 — Variant Calling: Pileup and Genotype Likelihoods

**Module:** 09 — Genomics and NGS  
**Notebook:** 07 of 16  
**Time estimate:** 90 minutes

> The math behind pileup-based SNP calling and genotype likelihoods is asked
> directly in bioinformatics interviews. You must be able to derive it.

---
## Step 1 — Motivation

Given a pile of reads at a position and their quality scores, how do we decide
whether the observed non-reference bases represent a real variant or a sequencing
error? Bayesian genotype likelihoods formalize this: we compute P(reads | genotype)
for each possible genotype and pick the most probable one.

---
## Step 2 — Intuition

**Pileup:** At each reference position, collect all reads covering that position.
Count the observed bases: how many A, C, G, T? High-quality non-reference bases
with strand balance and uniform positions suggest a real variant.

**Genotype likelihoods:** For a diploid organism at a position, possible genotypes
are AA, AC, AG, AT, CC, CG, CT, GG, GT, TT (10 combinations). For each genotype,
we compute the probability of observing the read data given that genotype.

**The key insight:** A sequencing error at Q30 has P = 10^(-3) = 0.001. If we see
20 reads with ALT at Q30, the probability that all are errors is 0.001^20 = 10^(-60).
That's effectively impossible — we're looking at a real variant.

---
## Step 3 — Biological Background

**Types of variants:**
- **SNP (Single Nucleotide Polymorphism):** One base changed (REF A → ALT T)
- **SNV (Single Nucleotide Variant):** More general term; used in somatic contexts
- **Indel:** Insertion or deletion of 1+ bases
- **MNP:** Multiple adjacent SNPs in phase
- **SV:** Structural variant (>50 bp deletion, inversion, translocation, CNV)

**Zygosity:**
- **Homozygous REF:** 0/0 — all reads show REF allele
- **Heterozygous:** 0/1 — ~50% REF, ~50% ALT reads
- **Homozygous ALT:** 1/1 — all reads show ALT allele

**Allele frequency expectations:**
- Germline het: VAF ≈ 0.5 (diploid)
- Germline hom ALT: VAF ≈ 1.0
- Somatic mutation (tumor purity 0.8, het): VAF ≈ 0.4
- Artifact: VAF < 0.05 with low-quality bases, strand bias

---
## Step 4 — Mathematical Explanation

**Genotype likelihood for a single base observation:**

For a diploid with genotype $G = (a_1, a_2)$ (e.g., A/T), the probability of
observing base $b$ with quality $Q$:

$$P(b \mid G) = \frac{P(b \mid a_1) + P(b \mid a_2)}{2}$$

where:
$$P(b \mid a) = \begin{cases} 1 - \epsilon & b = a \\ \epsilon / 3 & b \neq a \end{cases}$$

and $\epsilon = 10^{-Q/10}$ is the Phred-scaled error probability.

**For $n$ independent reads:**
$$P(\text{reads} \mid G) = \prod_{i=1}^{n} P(b_i \mid G)$$

**PHRED-scaled genotype likelihood (PL field in VCF):**
$$PL(G) = -10 \log_{10} P(\text{reads} \mid G)$$

Convention: PL is reported relative to the most likely genotype (PL=0 for best).

**Posterior genotype (with Hardy-Weinberg prior on allele frequency $f$):**
$$P(G \mid \text{reads}) \propto P(\text{reads} \mid G) \cdot P(G)$$

where $P(\text{A/A}) = (1-f)^2$, $P(\text{A/T}) = 2f(1-f)$, $P(\text{T/T}) = f^2$.

In [ ]:
# Step 6 — Python: pileup SNP caller with genotype likelihoods
import numpy as np
from dataclasses import dataclass
from itertools import combinations_with_replacement
from collections import Counter


BASES = ['A', 'C', 'G', 'T']


def phred_to_prob(q: int) -> float:
    return 10 ** (-q / 10)


def base_likelihood(observed: str, allele: str, error_prob: float) -> float:
    """P(observed base | allele) given sequencing error rate."""
    if observed == allele:
        return 1 - error_prob
    else:
        return error_prob / 3  # uniform error to other 3 bases


def genotype_likelihood(bases: list[str], quals: list[int], genotype: tuple[str, str]) -> float:
    """
    Compute P(reads | genotype) for a pileup at one position.
    bases: observed bases from reads (e.g., ['A','A','T','A'])
    quals: corresponding Phred Q scores
    genotype: (allele1, allele2) tuple, e.g., ('A', 'T')
    """
    log_lik = 0.0
    a1, a2 = genotype
    for b, q in zip(bases, quals):
        eps = phred_to_prob(q)
        p_b_given_g = 0.5 * base_likelihood(b, a1, eps) + 0.5 * base_likelihood(b, a2, eps)
        log_lik += np.log10(p_b_given_g + 1e-300)
    return log_lik  # log10 likelihood


def all_genotypes() -> list[tuple[str, str]]:
    return list(combinations_with_replacement(BASES, 2))


def call_genotype(bases: list[str], quals: list[int]) -> dict:
    """
    Call the most likely genotype at a pileup position.
    Returns {genotype, PL_dict, GQ}.
    """
    genotypes = all_genotypes()
    log_liks = {g: genotype_likelihood(bases, quals, g) for g in genotypes}

    # Best genotype
    best_g = max(log_liks, key=log_liks.get)
    best_ll = log_liks[best_g]

    # PL scores: relative to best, PHRED-scaled
    PL = {g: int(-10 * (ll - best_ll)) for g, ll in log_liks.items()}

    # Genotype quality (GQ): difference to second-best genotype
    sorted_PLs = sorted(PL.values())
    GQ = sorted_PLs[1] if len(sorted_PLs) > 1 else 99

    return {
        'genotype': best_g,
        'PL': PL,
        'GQ': min(GQ, 99),
        'log_lik': best_ll,
    }


@dataclass
class PileupPosition:
    chrom: str
    pos: int
    ref: str
    bases: list[str]
    quals: list[int]

    @property
    def depth(self) -> int:
        return len(self.bases)

    @property
    def base_counts(self) -> dict:
        return dict(Counter(self.bases))

    @property
    def alt_vaf(self) -> float:
        ref_count = self.bases.count(self.ref)
        return 1 - ref_count / self.depth if self.depth > 0 else 0.0


def pileup_snp_caller(
    pileup: list[PileupPosition],
    min_depth: int = 8,
    min_vaf: float = 0.15,
    min_gq: int = 20
) -> list[dict]:
    """
    Simple pileup-based SNP caller.
    Returns list of variant calls.
    """
    calls = []
    for pos in pileup:
        if pos.depth < min_depth:
            continue
        if pos.alt_vaf < min_vaf:
            continue  # all reads match REF
        result = call_genotype(pos.bases, pos.quals)
        gt = result['genotype']
        if gt == (pos.ref, pos.ref):  # homozygous reference
            continue
        if result['GQ'] < min_gq:
            continue
        alt = [a for a in set(gt) if a != pos.ref]
        if not alt:
            continue
        calls.append({
            'chrom': pos.chrom,
            'pos': pos.pos,
            'ref': pos.ref,
            'alt': alt[0],
            'depth': pos.depth,
            'vaf': pos.alt_vaf,
            'genotype': f'{gt[0]}/{gt[1]}',
            'GQ': result['GQ'],
            'counts': pos.base_counts,
        })
    return calls


# Simulate pileup data with known SNPs
rng = np.random.default_rng(42)
ref_seq = ''.join(rng.choice(['A','C','G','T'], 200))
snp_positions = {50: ('A', 'T'), 100: ('C', 'G'), 150: ('G', 'A')}  # pos: (REF, ALT)

pileup = []
for i, ref_base in enumerate(ref_seq):
    pos = i + 1
    depth = int(rng.integers(20, 40))
    if pos in snp_positions:
        ref_b, alt_b = snp_positions[pos]
        ref_base = ref_b
        n_alt = int(depth * 0.5)  # heterozygous
        n_ref = depth - n_alt
        bases = [ref_b] * n_ref + [alt_b] * n_alt
        rng.shuffle(bases)
        quals = [int(rng.integers(25, 38)) for _ in range(depth)]
    else:
        error_p = 0.005
        bases = []
        for _ in range(depth):
            if rng.random() < error_p:
                bases.append(rng.choice([b for b in BASES if b != ref_base]))
            else:
                bases.append(ref_base)
        quals = [int(rng.integers(28, 38)) for _ in range(depth)]
    pileup.append(PileupPosition('chr1', pos, ref_base, list(bases), quals))

variants = pileup_snp_caller(pileup)
print(f"{'Chrom':<8} {'Pos':>6} {'REF':>4} {'ALT':>4} {'Depth':>7} {'VAF':>6} {'GT':>6} {'GQ':>4}")
print('-' * 55)
for v in variants:
    print(f"{v['chrom']:<8} {v['pos']:>6} {v['ref']:>4} {v['alt']:>4} "
          f"{v['depth']:>7} {v['vaf']:>6.3f} {v['genotype']:>6} {v['GQ']:>4}")
print(f"\nTrue SNPs planted at positions: {sorted(snp_positions.keys())}")
called_positions = [v['pos'] for v in variants]
print(f"Called positions: {called_positions}")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Pileup at a SNP position
ax = axes[0, 0]
snp_pos = pileup[99]  # position 100 (0-indexed 99)
bc = snp_pos.base_counts
bars = ax.bar(bc.keys(), bc.values(), color=['#4FC3F7','#81C784','#FFB74D','#F06292'])
ax.set_xlabel('Base')
ax.set_ylabel('Count')
ax.set_title(f'A. Pileup at position {snp_pos.pos} (planted C→G SNP)\nDepth={snp_pos.depth}, VAF={snp_pos.alt_vaf:.2f}')
for bar, count in zip(bars, bc.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(count), ha='center')

# Panel B: Genotype likelihood landscape
ax2 = axes[0, 1]
snp_p = pileup[99]
genotypes = all_genotypes()
log_liks = [genotype_likelihood(snp_p.bases, snp_p.quals, g) for g in genotypes]
gt_labels = [f'{g[0]}/{g[1]}' for g in genotypes]
# Normalize to most likely = 0
max_ll = max(log_liks)
pl_vals = [-10 * (ll - max_ll) for ll in log_liks]
colors_gt = ['tomato' if pl == 0 else 'steelblue' for pl in pl_vals]
ax2.bar(gt_labels, [min(pl, 200) for pl in pl_vals], color=colors_gt)
ax2.set_xlabel('Genotype')
ax2.set_ylabel('PL (PHRED; 0 = best)')
ax2.set_title('B. Genotype likelihood landscape\n(best genotype has PL=0)')
ax2.tick_params(axis='x', rotation=60, labelsize=7)

# Panel C: VAF distribution across all pileup positions
ax3 = axes[1, 0]
vafs = [p.alt_vaf for p in pileup]
ax3.hist(vafs, bins=40, color='steelblue', alpha=0.8, edgecolor='none')
ax3.set_xlabel('Alt allele frequency (VAF)')
ax3.set_ylabel('Number of positions')
ax3.set_title('C. VAF distribution across all positions\n(SNPs visible at VAF~0.5)')
ax3.axvline(0.5, color='red', ls='--', label='Expected het VAF')
ax3.axvline(0.15, color='orange', ls=':', label='Min VAF filter')
ax3.legend(fontsize=8)

# Panel D: GQ vs depth
ax4 = axes[1, 1]
# Simulate GQ as a function of depth at VAF=0.5
depths = range(5, 60)
gqs = []
for d in depths:
    rng2 = np.random.default_rng(7)
    n_alt = d // 2
    bases = ['A'] * (d - n_alt) + ['T'] * n_alt
    quals = [30] * d
    result = call_genotype(bases, quals)
    gqs.append(result['GQ'])
ax4.plot(depths, gqs, 'b-o', ms=4, lw=2)
ax4.axhline(20, color='orange', ls='--', label='GQ=20 filter')
ax4.set_xlabel('Coverage depth')
ax4.set_ylabel('Genotype Quality (GQ)')
ax4.set_title('D. GQ increases with depth\n(VAF=0.5, Q=30 bases)')
ax4.legend(fontsize=8)

plt.suptitle('Pileup-based SNP Calling and Genotype Likelihoods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pileup_snp_calling.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. At a position with 20 reads, all showing base A with Q30. Compute the log10
   genotype likelihood for genotypes A/A, A/T, and T/T. Which is called?
2. Repeat for a position with 10 A reads and 10 T reads, all Q30. What is the GQ?
3. At what depth does GQ first exceed 20 for a het call at Q30 (VAF=0.5)? Compute
   by hand for depth 5, 8, 10.
4. Implement a prior that accounts for Hardy-Weinberg equilibrium (use allele
   frequency 0.001 for a rare SNP). How does this change the call for 1/10 reads
   supporting ALT?

---
## Step 10 — Quiz

1. Write the formula for P(base | genotype) for a diploid with genotype A/T.
2. What is a PL score? What does PL=0 mean? PL=100?
3. What is GQ and how is it derived from PL values?
4. Why does genotype calling require a minimum depth threshold?
5. A somatic variant has VAF=0.15. What minimum depth do you need to call it
   with at least 3 ALT reads?

---
## Step 12 — Reflection

> *[Explain in one paragraph: why do we need genotype likelihoods rather than just
> counting alt reads? Under what conditions would a simple count threshold fail?]*

---
*Next: `08_gatk_haplotypecaller_concepts.ipynb`*